# Урок 4. Подготовка признаков: модель наконец обгоняет правило

В уроках 1–3 модель застряла на ≈ 0.787 — на уровне правила «женщины выжили».
Здесь даём ей новую информацию (титул, размер семьи, порт) в понятном ей виде.

## 0. Данные и новые признаки

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate

df = pd.read_csv("../data/titanic.csv")
y = df["Survived"]

# Титул из имени: «Braund, Mr. Owen Harris» → «Mr»
df["Title"] = (df["Name"].str.split(", ").str[1]   # «Mr. Owen Harris»
                          .str.split(" ").str[0]     # «Mr.»
                          .str.rstrip("."))          # «Mr»
df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
df["Title"] = df["Title"].where(df["Title"].isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")

# Размер семьи: братья/сёстры/супруги + родители/дети + сам пассажир
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

df.groupby("Title")["Survived"].agg(["mean", "size"]).round(2)

,mean,size
Title,,
Master,0.57,40
Miss,0.70,185
Mr,0.16,517
Mrs,0.79,126
Rare,0.35,23


**Вывод:** Master — это мальчик (медиана возраста 3.5 года). Мальчики выживали в 57% случаев, взрослые мужчины (Mr) — только в 16%.
Титул известен у всех, даже у тех, у кого пропущен возраст.

## 1. One-hot: зачем

Класс, возраст, цена, размер семьи — это **числа**: их можно сравнивать «больше/меньше».
Пол, порт и титул только **делят людей на группы** — назвать их числами нельзя.
One-hot делает для каждого значения свою колонку из 0 и 1.
`handle_unknown="ignore"` — если встретится незнакомое значение, код не падает.

In [2]:
def make_model(num_cols, cat_cols):
    preprocess = ColumnTransformer([
        # числа: заполнить пропуски медианой → привести к одной шкале
        ("num", make_pipeline(SimpleImputer(strategy="median"),
                              StandardScaler()), num_cols),
        # категории: заполнить самым частым значением → one-hot
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                              OneHotEncoder(handle_unknown="ignore")), cat_cols),
    ])
    return make_pipeline(preprocess, LogisticRegression(max_iter=1000))

num_cols = ["Pclass", "Age", "Fare", "FamilySize"]
cat_cols = ["Sex", "Embarked", "Title"]
model = make_model(num_cols, cat_cols)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
r = cross_validate(model, df, y, cv=cv, scoring=["accuracy", "roc_auc"])
print(f"accuracy = {r['test_accuracy'].mean():.3f} ± {r['test_accuracy'].std():.3f}")
print(f"AUC      = {r['test_roc_auc'].mean():.3f}")

accuracy = 0.829 ± 0.009
AUC      = 0.874


**Вывод:** accuracy 0.829 — модель впервые уверенно обогнала правило по полу (0.787). Разница 4 пункта больше разброса (±0.009), значит, улучшение настоящее.

**Что такое AUC:** число от 0.5 до 1. Это вероятность того, что модель поставит случайного выжившего выше случайного погибшего. 0.5 = как угадывание, 1 = идеально. Здесь AUC = 0.874. В отличие от accuracy, AUC видит вероятности, а не только ответы «выжил/погиб».

## 2. Что видит модель: первые 3 пассажира после подготовки

In [3]:
preprocess = model[0]          # первый шаг pipeline — подготовка
preprocess.fit(df)
pd.DataFrame(preprocess.transform(df.head(3)),
             columns=preprocess.get_feature_names_out()).round(2)

,num__Pclass,num__Age,num__Fare,num__FamilySize,cat__Sex_female,cat__Sex_male,cat__Embarked_C,cat__Embarked_Q,cat__Embarked_S,cat__Title_Master,cat__Title_Miss,cat__Title_Mr,cat__Title_Mrs,cat__Title_Rare
0,0.83,-0.57,-0.50,0.06,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,-1.57,0.66,0.79,0.06,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0.83,-0.26,-0.49,-0.56,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


**Как читать:** у чисел после масштабирования плюс — больше среднего, минус — меньше, около 0 — средний.
У категорий единица показывает, к какой группе относится человек.

**Первая строка:** `Pclass` +0.83 → 3-й класс; `Age` −0.57 → моложе среднего (22 года); `Sex_male` = 1; `Embarked_S` = 1;
`FamilySize` 0.06 → почти средний (семья из 2 человек); **`Title_Mr` = 1 → взрослый мужчина** (не мальчик: мальчик был бы `Title_Master`).
Это Braund, Mr. Owen Harris, 22 года.

## 3. Таблица шаг за шагом: что даёт каждый признак

In [4]:
steps = [
    ("4 старых признака", ["Pclass", "Age", "Fare"],               ["Sex"]),
    ("+ порт",            ["Pclass", "Age", "Fare"],               ["Sex", "Embarked"]),
    ("+ титул",           ["Pclass", "Age", "Fare"],               ["Sex", "Embarked", "Title"]),
    ("+ размер семьи",    ["Pclass", "Age", "Fare", "FamilySize"], ["Sex", "Embarked", "Title"]),
]

rows = []
for name, num, cat in steps:
    r = cross_validate(make_model(num, cat), df, y, cv=cv, scoring=["accuracy", "roc_auc"])
    rows.append({"модель": name,
                 "accuracy": r["test_accuracy"].mean(),
                 "std": r["test_accuracy"].std(),
                 "AUC": r["test_roc_auc"].mean()})

pd.DataFrame(rows).round(3)

,модель,accuracy,std,AUC
0,4 старых признака,0.786,0.013,0.846
1,+ порт,0.789,0.020,0.848
2,+ титул,0.807,0.023,0.861
3,+ размер семьи,0.829,0.009,0.874


**Вывод:** порт почти ничего не дал, титул поднял accuracy с 0.789 до 0.807, размер семьи — до 0.829.

## 4. Ошибка: порт числами вместо one-hot

In [5]:
df["Emb_num"] = df["Embarked"].map({"S": 0, "C": 1, "Q": 2})

wrong = make_model(["Pclass", "Age", "Fare", "FamilySize", "Emb_num"], ["Sex", "Title"])
r = cross_validate(wrong, df, y, cv=cv, scoring=["accuracy", "roc_auc"])
print(f"порт числами: accuracy = {r['test_accuracy'].mean():.3f}, AUC = {r['test_roc_auc'].mean():.3f}")

df.groupby("Embarked")["Survived"].mean().round(2)

порт числами: accuracy = 0.828, AUC = 0.873


Embarked
C    0.55
Q    0.39
S    0.34

**Вывод:** результат почти тот же (0.828 против 0.829) — разница 0.001 меньше разброса, это шум.
Но так делать всё равно неправильно. Доля выживших: S = 0.34, C = 0.55, Q = 0.39.
С кодом S=0, C=1, Q=2 модель может нарисовать только прямую линию: если C лучше S, то Q обязан быть ещё лучше. А Q хуже C — модель не может это выучить.
С one-hot у каждого порта свои баллы. Здесь вреда мало, потому что порт слабый признак.

## 5. Вопрос: почему FamilySize можно в df, а StandardScaler — только в pipeline

**FamilySize** считается из строки самого человека (`SibSp + Parch + 1`) и не смотрит на других пассажиров. Учить ему нечего — утечки нет.

**StandardScaler** учится на данных: считает среднее и разброс по всем строкам. Если сделать это до разбиения, в среднее попадут тестовые люди — это утечка.
Поэтому он внутри pipeline: в каждом раунде кросс-валидации учится только на учебной части.
Он приводит числа к одной шкале (среднее 0, отклонение 1), поэтому коэффициенты можно сравнивать, и модели легче учиться.